# 05 - Gold: Modelo Analitico

## Objetivo

Construir o modelo dimensional em estrela utilizado para consumo analitico dos
Beneficios Concedidos pelo INSS, a partir da Gold preparatoria ja validada.

O modelo final e composto por:

- `afastamento_inss.gold.dim_tempo`
- `afastamento_inss.gold.dim_cid`
- `afastamento_inss.gold.dim_especie`
- `afastamento_inss.gold.dim_geografia`
- `afastamento_inss.gold.dim_atividade`
- `afastamento_inss.gold.fato_afastamentos`

## Grao do modelo

- A Gold preparatoria (`afastamento_inss.gold.prep_beneficios_bi`) possui uma
  linha por registro de beneficio concedido.
- A tabela fato possui uma linha por registro classificado como afastamento
  (`ind_afastamento = 1`).
- O modelo **nao** representa pessoas, beneficiarios ou trabalhadores unicos,
  pois a fonte nao possui identificador individual validado. Nenhuma dimensao
  ou indicador deste notebook infere unicidade de pessoas.
- O campo de municipio representa o municipio de **residencia**, nao o local
  de trabalho.
- O CNAE representa a atividade economica quando informada pela fonte, e nao
  uma condicao de trabalho. A ausencia de CNAE nao significa que a pessoa nao
  trabalha.

## Convencoes

- Chaves subrogadas (`_sk`) sao deterministicas, geradas com `sha2` sobre os
  atributos de negocio, nunca com `monotonically_increasing_id`.
- Toda dimensao possui um membro padrao `NI` para cobrir ausencia ou
  invalidez do atributo de origem, garantindo integridade referencial total
  com a fato.
- Nenhuma gravacao e executada apos uma validacao critica ter falhado; toda
  falha critica interrompe o notebook com `raise ValueError`.


In [0]:
# ============================================================
# IMPORTACOES
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    when,
    concat_ws,
    sha2,
    count,
    countDistinct,
    sum as spark_sum,
    current_timestamp,
    regexp_extract,
    trim,
    coalesce,
    row_number
)

from pyspark.sql.window import Window


In [0]:
# ============================================================
# PARAMETROS
# ============================================================

TABELA_ORIGEM = "afastamento_inss.gold.prep_beneficios_bi"
TABELA_DIM_TEMPO = "afastamento_inss.gold.dim_tempo"
TABELA_DIM_CID = "afastamento_inss.gold.dim_cid"
TABELA_DIM_ESPECIE = "afastamento_inss.gold.dim_especie"
TABELA_DIM_GEOGRAFIA = "afastamento_inss.gold.dim_geografia"
TABELA_DIM_ATIVIDADE = "afastamento_inss.gold.dim_atividade"
TABELA_FATO = "afastamento_inss.gold.fato_afastamentos"

print(f"Origem              : {TABELA_ORIGEM}")
print(f"Dimensao tempo      : {TABELA_DIM_TEMPO}")
print(f"Dimensao CID        : {TABELA_DIM_CID}")
print(f"Dimensao especie    : {TABELA_DIM_ESPECIE}")
print(f"Dimensao geografia  : {TABELA_DIM_GEOGRAFIA}")
print(f"Dimensao atividade  : {TABELA_DIM_ATIVIDADE}")
print(f"Fato                : {TABELA_FATO}")


## 1. Leitura e validacao da Gold preparatoria

O modelo dimensional e construido exclusivamente a partir da Gold
preparatoria validada. Isso centraliza as regras semanticas ja aplicadas e
evita que as dimensoes ou a fato reconstruam tratamentos ja realizados
upstream.

O DataFrame de origem e mantido em cache durante toda a execucao do
notebook e liberado explicitamente ao final.

In [0]:
# ============================================================
# LEITURA E CACHE DA GOLD PREPARATORIA
# ============================================================

df_prep = spark.table(TABELA_ORIGEM)

quantidade_linhas_prep = df_prep.count()
quantidade_colunas_prep = len(df_prep.columns)

print(f"Linhas  : {quantidade_linhas_prep:,}")
print(f"Colunas : {quantidade_colunas_prep}")


In [0]:
# ============================================================
# VALIDACAO DO CONTRATO DE ENTRADA
# ============================================================

COLUNAS_OBRIGATORIAS = [
    "competencia_concessao",
    "dt_competencia",
    "ano_competencia",
    "mes_competencia",
    "ano_mes_competencia",
    "cid_cod",
    "cid_desc",
    "cid_capitulo",
    "cid_grupo",
    "cid_grupo_desc",
    "cid_status",
    "cid_status_desc",
    "especie_cod",
    "especie_desc",
    "tipo_beneficio",
    "tipo_beneficio_desc",
    "natureza_afastamento",
    "natureza_afastamento_desc",
    "mun_resid",
    "mun_cod",
    "mun_nome",
    "uf",
    "ramo_atividade",
    "cnae_2023",
    "cnae_2024",
    "dt_nascimento",
    "idade_na_competencia",
    "faixa_etaria",
    "sexo",
    "clientela",
    "forma_filiacao",
    "grau_instrucao",
    "qt_anos_contribuicao",
    "qt_sm_rmi",
    "dt_dib",
    "dt_ddb",
    "dt_dcb",
    "duracao_beneficio_dias",
    "qtd_beneficios",
    "ind_afastamento",
    "ind_cid_informado",
    "ind_saude_mental",
    "ind_osteomuscular",
    "ind_cardiovascular",
    "ind_respiratorio",
    "ind_acidentario",
    "ind_afastamento_saude_mental",
    "ind_afastamento_osteomuscular",
    "ind_afastamento_cardiovascular",
    "ind_afastamento_respiratorio"
]

colunas_ausentes = [
    nome for nome in COLUNAS_OBRIGATORIAS
    if nome not in df_prep.columns
]

if colunas_ausentes:
    raise ValueError(
        "Colunas obrigatorias ausentes na Gold preparatoria: "
        + ", ".join(colunas_ausentes)
    )

print("Contrato de entrada validado.")
print(f"Colunas obrigatorias verificadas: {len(COLUNAS_OBRIGATORIAS)}")


## 2. Dimensao de tempo

Grao da dimensao:

> Uma linha por competencia de concessao distinta.

A chave `tempo_sk` e gerada de forma deterministica com SHA-256 sobre
`competencia_concessao`.

In [0]:
# ============================================================
# DIMENSAO DE TEMPO
# ============================================================

df_dim_tempo = (
    df_prep
    .filter(col("competencia_concessao").isNotNull())
    .select(
        "competencia_concessao",
        "dt_competencia",
        "ano_competencia",
        "mes_competencia",
        "ano_mes_competencia"
    )
    .distinct()
    .withColumn(
        "tempo_sk",
        sha2(col("competencia_concessao"), 256)
    )
    .withColumn(
        "nome_mes",
        when(col("mes_competencia") == 1, lit("Janeiro"))
        .when(col("mes_competencia") == 2, lit("Fevereiro"))
        .when(col("mes_competencia") == 3, lit("Marco"))
        .when(col("mes_competencia") == 4, lit("Abril"))
        .when(col("mes_competencia") == 5, lit("Maio"))
        .when(col("mes_competencia") == 6, lit("Junho"))
        .when(col("mes_competencia") == 7, lit("Julho"))
        .when(col("mes_competencia") == 8, lit("Agosto"))
        .when(col("mes_competencia") == 9, lit("Setembro"))
        .when(col("mes_competencia") == 10, lit("Outubro"))
        .when(col("mes_competencia") == 11, lit("Novembro"))
        .when(col("mes_competencia") == 12, lit("Dezembro"))
        .otherwise(lit("Mes nao mapeado"))
    )
    .withColumn(
        "trimestre",
        when(col("mes_competencia").between(1, 3), lit("1o trimestre"))
        .when(col("mes_competencia").between(4, 6), lit("2o trimestre"))
        .when(col("mes_competencia").between(7, 9), lit("3o trimestre"))
        .when(col("mes_competencia").between(10, 12), lit("4o trimestre"))
        .otherwise(lit("Nao mapeado"))
    )
    .withColumn(
        "semestre",
        when(col("mes_competencia").between(1, 6), lit("1o semestre"))
        .when(col("mes_competencia").between(7, 12), lit("2o semestre"))
        .otherwise(lit("Nao mapeado"))
    )
    .withColumn("_data_processamento", current_timestamp())
    .select(
        "tempo_sk",
        "competencia_concessao",
        "dt_competencia",
        "ano_competencia",
        "mes_competencia",
        "nome_mes",
        "trimestre",
        "semestre",
        "ano_mes_competencia",
        "_data_processamento"
    )
)

display(df_dim_tempo)


In [0]:
# ============================================================
# VALIDACAO DA DIMENSAO DE TEMPO
# ============================================================

total_dim_tempo = df_dim_tempo.count()

chaves_tempo_distintas = (
    df_dim_tempo.select("tempo_sk").distinct().count()
)

competencias_distintas_origem = (
    df_prep
    .filter(col("competencia_concessao").isNotNull())
    .select("competencia_concessao")
    .distinct()
    .count()
)

chaves_tempo_nulas = (
    df_dim_tempo.filter(col("tempo_sk").isNull()).count()
)

print("=" * 65)
print("VALIDACAO DA DIMENSAO DE TEMPO")
print("=" * 65)
print(f"{'Registros da dimensao':40}{total_dim_tempo:>12,}")
print(f"{'Chaves distintas':40}{chaves_tempo_distintas:>12,}")
print(f"{'Competencias distintas na origem':40}{competencias_distintas_origem:>12,}")
print(f"{'Chaves nulas':40}{chaves_tempo_nulas:>12,}")
print("-" * 65)

erros_dim_tempo = []

if total_dim_tempo != chaves_tempo_distintas:
    erros_dim_tempo.append("A chave tempo_sk nao e unica.")

if total_dim_tempo != competencias_distintas_origem:
    erros_dim_tempo.append(
        "A quantidade de competencias diverge da origem."
    )

if chaves_tempo_nulas > 0:
    erros_dim_tempo.append("Existem chaves tempo_sk nulas.")

if not erros_dim_tempo:
    print("RESULTADO: OK")
    print(
        "A dimensao possui uma linha por competencia "
        "e chave tecnica unica."
    )
else:
    print("RESULTADO: FALHA")
    for erro in erros_dim_tempo:
        print(f"- {erro}")

print("=" * 65)

if erros_dim_tempo:
    raise ValueError(
        "Validacao da dimensao de tempo falhou: "
        + " | ".join(erros_dim_tempo)
    )


In [0]:
# ============================================================
# GRAVACAO DA DIMENSAO DE TEMPO
# ============================================================

(
    df_dim_tempo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_TEMPO)
)

print(f"Tabela gravada: {TABELA_DIM_TEMPO}")


In [0]:
# ============================================================
# VALIDACAO DA PERSISTENCIA DA DIMENSAO DE TEMPO
# ============================================================

df_dim_tempo_persistida = spark.table(TABELA_DIM_TEMPO)

linhas_tempo_antes = df_dim_tempo.count()
linhas_tempo_depois = df_dim_tempo_persistida.count()

print("=" * 65)
print("VALIDACAO DA PERSISTENCIA DA DIMENSAO TEMPO")
print("=" * 65)
print(f"{'Antes da gravacao':35}{linhas_tempo_antes:>12,}")
print(f"{'Depois da gravacao':35}{linhas_tempo_depois:>12,}")
print("-" * 65)

if linhas_tempo_antes == linhas_tempo_depois:
    print("RESULTADO: OK")
    print("A dimensao foi persistida sem perda de registros.")
else:
    print("RESULTADO: FALHA")
    print("A persistencia alterou a quantidade de registros.")

print("=" * 65)

if linhas_tempo_antes != linhas_tempo_depois:
    raise ValueError(
        "Persistencia da dimensao de tempo alterou a quantidade "
        "de registros."
    )

display(df_dim_tempo_persistida)


In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.dim_tempo
IS 'Dimensao de tempo do modelo analitico de beneficios e afastamentos do INSS. Possui uma linha por competencia de concessao distinta.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN tempo_sk
COMMENT 'Chave tecnica deterministica da competencia, gerada por SHA-256 sobre competencia_concessao.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN competencia_concessao
COMMENT 'Competencia de concessao do beneficio, conforme informada pela fonte.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN ano_competencia
COMMENT 'Ano da competencia de concessao.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN mes_competencia
COMMENT 'Mes da competencia de concessao, numerico de 1 a 12.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN nome_mes
COMMENT 'Nome do mes da competencia de concessao.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN trimestre
COMMENT 'Trimestre calendario correspondente ao mes da competencia.';

ALTER TABLE afastamento_inss.gold.dim_tempo
ALTER COLUMN semestre
COMMENT 'Semestre calendario correspondente ao mes da competencia.';


## 3. Dimensao CID

Grao da dimensao:

> Uma linha por codigo CID distinto.

Registros sem CID recebem um membro padrao:

- codigo tecnico `NI`;
- descricao `Diagnostico nao informado`.

Codigos preenchidos cuja descricao nao foi disponibilizada pela fonte
recebem `Descricao nao disponibilizada pela fonte`, sem preenchimento
artificial de conteudo. Nenhuma descricao oficial e inventada.

Antes da criacao da dimensao, o notebook valida se um mesmo codigo CID
possui classificacoes conflitantes de descricao, capitulo, grupo ou status.
Havendo conflito, a execucao e interrompida.

In [0]:
# ============================================================
# DIMENSAO CID - BASE E ATRIBUTOS MODELO
# ============================================================

df_cid_base = (
    df_prep
    .select(
        "cid_cod",
        "cid_desc",
        "cid_capitulo",
        "cid_grupo",
        "cid_grupo_desc",
        "cid_status",
        "cid_status_desc"
    )
    .withColumn(
        "cid_cod_modelo",
        when(col("cid_cod").isNull(), lit("NI")).otherwise(col("cid_cod"))
    )
    .withColumn(
        "cid_desc_modelo",
        when(col("cid_cod").isNull(), lit("Diagnostico nao informado"))
        .when(col("cid_desc").isNull(), lit("Descricao nao disponibilizada pela fonte"))
        .otherwise(col("cid_desc"))
    )
    .withColumn(
        "cid_capitulo_modelo",
        when(col("cid_cod").isNull(), lit("NI")).otherwise(col("cid_capitulo"))
    )
    .withColumn(
        "cid_grupo_modelo",
        when(col("cid_cod").isNull(), lit("NI")).otherwise(col("cid_grupo"))
    )
    .withColumn(
        "cid_status_modelo",
        when(col("cid_cod").isNull(), lit("NI")).otherwise(col("cid_status"))
    )
)


In [0]:
# ============================================================
# DIMENSAO CID - DIAGNOSTICO E VALIDACAO DE CONFLITOS
# ============================================================

df_conflitos_cid = (
    df_cid_base
    .groupBy("cid_cod_modelo")
    .agg(
        countDistinct("cid_desc_modelo").alias("qtd_descricoes_distintas"),
        countDistinct("cid_capitulo_modelo").alias("qtd_capitulos_distintos"),
        countDistinct("cid_grupo_modelo").alias("qtd_grupos_distintos"),
        countDistinct("cid_status_modelo").alias("qtd_status_distintos")
    )
    .filter(
        (col("qtd_descricoes_distintas") > 1)
        | (col("qtd_capitulos_distintos") > 1)
        | (col("qtd_grupos_distintos") > 1)
        | (col("qtd_status_distintos") > 1)
    )
)

quantidade_codigos_com_conflito = df_conflitos_cid.count()

df_cid_sem_descricao = (
    df_prep
    .filter(col("cid_cod").isNotNull() & col("cid_desc").isNull())
    .groupBy("cid_cod", "cid_capitulo", "cid_grupo", "cid_grupo_desc")
    .count()
    .orderBy("count", ascending=False)
)

quantidade_registros_sem_descricao = (
    df_prep
    .filter(col("cid_cod").isNotNull() & col("cid_desc").isNull())
    .count()
)

print("=" * 70)
print("DIAGNOSTICO E CONFLITOS DA DIMENSAO CID")
print("=" * 70)
print(f"{'Codigos com classificacao conflitante':45}{quantidade_codigos_com_conflito:>12,}")
print(f"{'Registros com codigo preenchido sem descricao':45}{quantidade_registros_sem_descricao:>12,}")
print(f"{'Codigos distintos sem descricao':45}{df_cid_sem_descricao.count():>12,}")
print("=" * 70)

display(df_conflitos_cid)
display(df_cid_sem_descricao)

if quantidade_codigos_com_conflito > 0:
    raise ValueError(
        "Existem codigos CID com classificacoes conflitantes de "
        "descricao, capitulo, grupo ou status. Execucao interrompida."
    )


In [0]:
# ============================================================
# DIMENSAO CID - CRIACAO
# ============================================================

df_dim_cid = (
    df_cid_base
    .select(
        "cid_cod_modelo",
        "cid_desc_modelo",
        "cid_capitulo_modelo",
        "cid_grupo_modelo",
        "cid_grupo_desc",
        "cid_status_modelo",
        "cid_status_desc"
    )
    .distinct()
    .withColumn("cid_sk", sha2(col("cid_cod_modelo"), 256))
    .withColumn(
        "descricao_fornecida_fonte",
        when(col("cid_cod_modelo") == "NI", lit(0))
        .when(col("cid_desc_modelo") == "Descricao nao disponibilizada pela fonte", lit(0))
        .otherwise(lit(1))
    )
    .withColumn("_data_processamento", current_timestamp())
    .select(
        "cid_sk",
        col("cid_cod_modelo").alias("cid_cod"),
        col("cid_desc_modelo").alias("cid_desc"),
        col("cid_capitulo_modelo").alias("cid_capitulo"),
        col("cid_grupo_modelo").alias("cid_grupo"),
        "cid_grupo_desc",
        col("cid_status_modelo").alias("cid_status"),
        "cid_status_desc",
        "descricao_fornecida_fonte",
        "_data_processamento"
    )
)

display(df_dim_cid.orderBy("cid_cod"))


In [0]:
# ============================================================
# DIMENSAO CID - VALIDACAO DA DIMENSAO
# ============================================================

total_dim_cid = df_dim_cid.count()

chaves_cid_distintas = df_dim_cid.select("cid_sk").distinct().count()

codigos_cid_distintos = df_dim_cid.select("cid_cod").distinct().count()

chaves_cid_nulas = df_dim_cid.filter(col("cid_sk").isNull()).count()

membros_nao_informados = df_dim_cid.filter(col("cid_cod") == "NI").count()

descricoes_nao_fornecidas = (
    df_dim_cid.filter(col("descricao_fornecida_fonte") == 0).count()
)

print("=" * 70)
print("VALIDACAO DA DIMENSAO CID")
print("=" * 70)
print(f"{'Registros da dimensao':45}{total_dim_cid:>12,}")
print(f"{'Chaves distintas':45}{chaves_cid_distintas:>12,}")
print(f"{'Codigos distintos':45}{codigos_cid_distintos:>12,}")
print(f"{'Chaves nulas':45}{chaves_cid_nulas:>12,}")
print(f"{'Membros padrao NI':45}{membros_nao_informados:>12,}")
print(f"{'Membros sem descricao da fonte':45}{descricoes_nao_fornecidas:>12,}")
print("-" * 70)

erros_dim_cid = []

if total_dim_cid != chaves_cid_distintas:
    erros_dim_cid.append("A chave cid_sk nao e unica.")

if total_dim_cid != codigos_cid_distintos:
    erros_dim_cid.append("Existe mais de uma linha para o mesmo codigo CID.")

if chaves_cid_nulas > 0:
    erros_dim_cid.append("Existem chaves cid_sk nulas.")

if membros_nao_informados != 1:
    erros_dim_cid.append("A dimensao deve possuir exatamente um membro padrao NI.")

if not erros_dim_cid:
    print("RESULTADO: OK")
    print(
        "A dimensao possui uma linha por codigo CID e um membro "
        "padrao para diagnostico nao informado."
    )
else:
    print("RESULTADO: FALHA")
    for erro in erros_dim_cid:
        print(f"- {erro}")

print("=" * 70)

if erros_dim_cid:
    raise ValueError(
        "Validacao da dimensao CID falhou: " + " | ".join(erros_dim_cid)
    )


In [0]:
# ============================================================
# DIMENSAO CID - VALIDACAO DE COBERTURA
# ============================================================

df_prep_com_cid_modelo = (
    df_prep
    .withColumn(
        "cid_cod_modelo",
        when(col("cid_cod").isNull(), lit("NI")).otherwise(col("cid_cod"))
    )
)

df_teste_cobertura_cid = (
    df_prep_com_cid_modelo.alias("f")
    .join(
        df_dim_cid.alias("d"),
        col("f.cid_cod_modelo") == col("d.cid_cod"),
        "left"
    )
)

registros_apos_join_cid = df_teste_cobertura_cid.count()

registros_sem_correspondencia_cid = (
    df_teste_cobertura_cid.filter(col("d.cid_sk").isNull()).count()
)

print("=" * 70)
print("VALIDACAO DE COBERTURA DA DIMENSAO CID")
print("=" * 70)
print(f"{'Registros da origem':45}{quantidade_linhas_prep:>12,}")
print(f"{'Registros apos relacionamento':45}{registros_apos_join_cid:>12,}")
print(f"{'Registros sem correspondencia':45}{registros_sem_correspondencia_cid:>12,}")
print("-" * 70)

erros_cobertura_cid = []

if registros_apos_join_cid != quantidade_linhas_prep:
    erros_cobertura_cid.append(
        "O relacionamento com a dimensao CID multiplicou ou removeu registros."
    )

if registros_sem_correspondencia_cid > 0:
    erros_cobertura_cid.append(
        "Existem registros sem correspondencia na dimensao CID."
    )

if not erros_cobertura_cid:
    print("RESULTADO: OK")
    print("Todos os registros encontram correspondencia na dimensao CID.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_cobertura_cid:
        print(f"- {erro}")

print("=" * 70)

if erros_cobertura_cid:
    raise ValueError(
        "Validacao de cobertura da dimensao CID falhou: "
        + " | ".join(erros_cobertura_cid)
    )


In [0]:
# ============================================================
# DIMENSAO CID - GRAVACAO
# ============================================================

(
    df_dim_cid
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_CID)
)

print(f"Tabela gravada: {TABELA_DIM_CID}")


In [0]:
# ============================================================
# DIMENSAO CID - VALIDACAO DA PERSISTENCIA
# ============================================================

df_dim_cid_persistida = spark.table(TABELA_DIM_CID)

linhas_cid_antes = df_dim_cid.count()
linhas_cid_depois = df_dim_cid_persistida.count()

print("=" * 70)
print("VALIDACAO DA PERSISTENCIA DA DIMENSAO CID")
print("=" * 70)
print(f"{'Antes da gravacao':40}{linhas_cid_antes:>12,}")
print(f"{'Depois da gravacao':40}{linhas_cid_depois:>12,}")
print("-" * 70)

if linhas_cid_antes == linhas_cid_depois:
    print("RESULTADO: OK")
    print("A dimensao CID foi persistida corretamente.")
else:
    print("RESULTADO: FALHA")
    print("A persistencia alterou a quantidade de registros.")

print("=" * 70)

if linhas_cid_antes != linhas_cid_depois:
    raise ValueError(
        "Persistencia da dimensao CID alterou a quantidade de registros."
    )


In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.dim_cid
IS 'Dimensao de diagnosticos CID utilizada no modelo analitico de beneficios e afastamentos. Possui uma linha por codigo CID distinto e um membro padrao para diagnostico nao informado.';

ALTER TABLE afastamento_inss.gold.dim_cid
ALTER COLUMN cid_sk
COMMENT 'Chave tecnica deterministica da dimensao CID, gerada por SHA-256.';

ALTER TABLE afastamento_inss.gold.dim_cid
ALTER COLUMN cid_cod
COMMENT 'Codigo CID informado pela fonte ou NI para diagnostico nao informado.';

ALTER TABLE afastamento_inss.gold.dim_cid
ALTER COLUMN cid_desc
COMMENT 'Descricao do CID disponibilizada pela fonte ou classificacao tecnica de ausencia.';

ALTER TABLE afastamento_inss.gold.dim_cid
ALTER COLUMN cid_grupo
COMMENT 'Grupo analitico do CID: mental, osteomuscular, cardiovascular, respiratorio, outros ou nao informado.';

ALTER TABLE afastamento_inss.gold.dim_cid
ALTER COLUMN descricao_fornecida_fonte
COMMENT 'Indicador binario: 1 quando a descricao foi disponibilizada pela fonte e 0 nos demais casos.';


## 4. Dimensao de especie

Grao da dimensao:

> Uma linha por codigo de especie distinto.

A dimensao preserva codigo e descricao da especie, o tipo analitico do
beneficio e a natureza do afastamento. A execucao e interrompida se um
mesmo codigo de especie possuir mais de uma descricao ou classificacao.

A classificacao descreve os registros de beneficio segundo a especie
informada pela fonte; ela nao representa pessoas unicas.

In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - BASE
# ============================================================

df_especie_base = (
    df_prep
    .select(
        "especie_cod",
        "especie_desc",
        "tipo_beneficio",
        "tipo_beneficio_desc",
        "natureza_afastamento",
        "natureza_afastamento_desc"
    )
)

print(f"Registros na base de especie: {df_especie_base.count():,}")


In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - DIAGNOSTICO E CONFLITOS
# ============================================================

especie_cod_nulo = df_especie_base.filter(col("especie_cod").isNull()).count()
especie_desc_nula = df_especie_base.filter(col("especie_desc").isNull()).count()

df_conflitos_descricao_especie = (
    df_especie_base
    .groupBy("especie_cod")
    .agg(countDistinct("especie_desc").alias("qtd_descricoes_distintas"))
    .filter(col("qtd_descricoes_distintas") > 1)
)

qtd_conflitos_descricao = df_conflitos_descricao_especie.count()

df_conflitos_classificacao_especie = (
    df_especie_base
    .groupBy("especie_cod")
    .agg(
        countDistinct("tipo_beneficio").alias("qtd_tipos_beneficio"),
        countDistinct("natureza_afastamento").alias("qtd_naturezas_afastamento")
    )
    .filter(
        (col("qtd_tipos_beneficio") > 1)
        | (col("qtd_naturezas_afastamento") > 1)
    )
)

qtd_conflitos_classificacao = df_conflitos_classificacao_especie.count()

df_dominio_especie = df_especie_base.distinct().orderBy("especie_cod")

naturezas_especie_31 = [
    linha["natureza_afastamento"]
    for linha in df_dominio_especie
        .filter(col("especie_cod") == "31")
        .select("natureza_afastamento")
        .distinct()
        .collect()
]

naturezas_especie_91 = [
    linha["natureza_afastamento"]
    for linha in df_dominio_especie
        .filter(col("especie_cod") == "91")
        .select("natureza_afastamento")
        .distinct()
        .collect()
]

print("=" * 70)
print("DIAGNOSTICO E CONFLITOS DA DIMENSAO DE ESPECIE")
print("=" * 70)
print(f"{'Registros com especie_cod nulo':45}{especie_cod_nulo:>12,}")
print(f"{'Registros com especie_desc nula':45}{especie_desc_nula:>12,}")
print(f"{'Codigos com mais de uma descricao':45}{qtd_conflitos_descricao:>12,}")
print(f"{'Codigos com classificacao conflitante':45}{qtd_conflitos_classificacao:>12,}")
print(f"{'Combinacoes distintas na base':45}{df_dominio_especie.count():>12,}")
print(f"Natureza da especie 31 (previdenciario esperado): {naturezas_especie_31}")
print(f"Natureza da especie 91 (acidentario esperado)   : {naturezas_especie_91}")
print("=" * 70)

display(df_dominio_especie)

erros_diagnostico_especie = []

if especie_cod_nulo > 0:
    erros_diagnostico_especie.append("Existem registros com especie_cod nulo.")

if especie_desc_nula > 0:
    erros_diagnostico_especie.append("Existem registros com especie_desc nula.")

if qtd_conflitos_descricao > 0:
    erros_diagnostico_especie.append(
        "Existem codigos de especie com mais de uma descricao."
    )

if qtd_conflitos_classificacao > 0:
    erros_diagnostico_especie.append(
        "Existem codigos de especie com classificacao conflitante."
    )

if naturezas_especie_31 and naturezas_especie_31 != ["previdenciario"]:
    erros_diagnostico_especie.append(
        "A especie 31 nao esta classificada exclusivamente como "
        "afastamento previdenciario."
    )

if naturezas_especie_91 and naturezas_especie_91 != ["acidentario"]:
    erros_diagnostico_especie.append(
        "A especie 91 nao esta classificada exclusivamente como "
        "afastamento acidentario."
    )

if erros_diagnostico_especie:
    raise ValueError(
        "Diagnostico da dimensao de especie encontrou inconsistencias: "
        + " | ".join(erros_diagnostico_especie)
    )


In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - CRIACAO
# ============================================================

df_dim_especie = (
    df_especie_base
    .distinct()
    .withColumn("especie_sk", sha2(col("especie_cod"), 256))
    .withColumn(
        "ind_especie_afastamento",
        when(col("tipo_beneficio") == "afastamento", lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "ind_especie_acidentaria",
        when(col("natureza_afastamento") == "acidentario", lit(1)).otherwise(lit(0))
    )
    .withColumn("_data_processamento", current_timestamp())
    .select(
        "especie_sk",
        "especie_cod",
        "especie_desc",
        "tipo_beneficio",
        "tipo_beneficio_desc",
        "natureza_afastamento",
        "natureza_afastamento_desc",
        "ind_especie_afastamento",
        "ind_especie_acidentaria",
        "_data_processamento"
    )
)

display(df_dim_especie.orderBy("especie_cod"))


In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - VALIDACAO
# ============================================================

total_dim_especie = df_dim_especie.count()

chaves_especie_distintas = df_dim_especie.select("especie_sk").distinct().count()

codigos_especie_distintos = df_dim_especie.select("especie_cod").distinct().count()

chaves_especie_nulas = df_dim_especie.filter(col("especie_sk").isNull()).count()

print("=" * 70)
print("VALIDACAO DA DIMENSAO DE ESPECIE")
print("=" * 70)
print(f"{'Registros da dimensao':45}{total_dim_especie:>12,}")
print(f"{'Chaves distintas':45}{chaves_especie_distintas:>12,}")
print(f"{'Codigos distintos':45}{codigos_especie_distintos:>12,}")
print(f"{'Chaves nulas':45}{chaves_especie_nulas:>12,}")
print("-" * 70)

erros_dim_especie = []

if total_dim_especie != chaves_especie_distintas:
    erros_dim_especie.append("A chave especie_sk nao e unica.")

if total_dim_especie != codigos_especie_distintos:
    erros_dim_especie.append("Existe mais de uma linha para o mesmo codigo.")

if chaves_especie_nulas > 0:
    erros_dim_especie.append("Existem chaves especie_sk nulas.")

if not erros_dim_especie:
    print("RESULTADO: OK")
    print("A dimensao possui uma linha por especie e chave tecnica unica.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_dim_especie:
        print(f"- {erro}")

print("=" * 70)

if erros_dim_especie:
    raise ValueError(
        "Validacao da dimensao de especie falhou: "
        + " | ".join(erros_dim_especie)
    )


In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - VALIDACAO DE COBERTURA
# ============================================================

df_teste_cobertura_especie = (
    df_prep.alias("f")
    .join(
        df_dim_especie.alias("d"),
        col("f.especie_cod") == col("d.especie_cod"),
        "left"
    )
)

registros_apos_join_especie = df_teste_cobertura_especie.count()

registros_sem_especie = (
    df_teste_cobertura_especie.filter(col("d.especie_sk").isNull()).count()
)

print("=" * 70)
print("VALIDACAO DE COBERTURA DA DIMENSAO DE ESPECIE")
print("=" * 70)
print(f"{'Registros na origem':45}{quantidade_linhas_prep:>12,}")
print(f"{'Registros apos relacionamento':45}{registros_apos_join_especie:>12,}")
print(f"{'Registros sem correspondencia':45}{registros_sem_especie:>12,}")
print("-" * 70)

erros_cobertura_especie = []

if registros_apos_join_especie != quantidade_linhas_prep:
    erros_cobertura_especie.append(
        "O relacionamento multiplicou ou removeu registros."
    )

if registros_sem_especie > 0:
    erros_cobertura_especie.append("Existem registros sem correspondencia.")

if not erros_cobertura_especie:
    print("RESULTADO: OK")
    print("Todos os registros encontram uma unica correspondencia na dimensao.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_cobertura_especie:
        print(f"- {erro}")

print("=" * 70)

if erros_cobertura_especie:
    raise ValueError(
        "Validacao de cobertura da dimensao de especie falhou: "
        + " | ".join(erros_cobertura_especie)
    )


In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - GRAVACAO
# ============================================================

(
    df_dim_especie
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_ESPECIE)
)

print(f"Tabela gravada: {TABELA_DIM_ESPECIE}")


In [0]:
# ============================================================
# DIMENSAO DE ESPECIE - VALIDACAO DA PERSISTENCIA
# ============================================================

df_dim_especie_persistida = spark.table(TABELA_DIM_ESPECIE)

linhas_especie_antes = df_dim_especie.count()
linhas_especie_depois = df_dim_especie_persistida.count()

print("=" * 70)
print("VALIDACAO DA PERSISTENCIA DA DIMENSAO DE ESPECIE")
print("=" * 70)
print(f"{'Antes da gravacao':40}{linhas_especie_antes:>12,}")
print(f"{'Depois da gravacao':40}{linhas_especie_depois:>12,}")
print("-" * 70)

if linhas_especie_antes == linhas_especie_depois:
    print("RESULTADO: OK")
    print("A dimensao de especie foi persistida sem alteracao de registros.")
else:
    print("RESULTADO: FALHA")
    print("A persistencia alterou a quantidade de registros.")

print("=" * 70)

if linhas_especie_antes != linhas_especie_depois:
    raise ValueError(
        "Persistencia da dimensao de especie alterou a quantidade "
        "de registros."
    )


In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.dim_especie
IS 'Dimensao de especies de beneficios do INSS. Possui uma linha por codigo de especie e centraliza classificacoes de tipo de beneficio e natureza do afastamento.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN especie_sk
COMMENT 'Chave tecnica deterministica da especie, gerada por SHA-256.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN especie_cod
COMMENT 'Codigo da especie do beneficio informado pela fonte.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN especie_desc
COMMENT 'Descricao da especie do beneficio informada pela fonte.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN tipo_beneficio
COMMENT 'Classificacao analitica do beneficio em afastamento, aposentadoria, pensao, assistencial, maternidade ou outros.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN natureza_afastamento
COMMENT 'Natureza administrativa do afastamento: previdenciario, acidentario, outras modalidades ou nao aplicavel.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN ind_especie_afastamento
COMMENT 'Indicador binario igual a 1 para especies classificadas como afastamento.';

ALTER TABLE afastamento_inss.gold.dim_especie
ALTER COLUMN ind_especie_acidentaria
COMMENT 'Indicador binario igual a 1 para especies classificadas como acidentarias.';


## 5. Dimensao geografica

Grao da dimensao:

> Uma linha por combinacao geografica distinta (codigo de municipio e sigla
> de UF apos normalizacao).

O campo original de municipio (`mun_resid`) segue o padrao
`codigo-UF-municipio` (exemplo: `21504-SP-Sao Paulo`). A dimensao extrai
codigo do municipio, sigla da UF e nome do municipio a partir desse campo, e
preserva o valor original para rastreabilidade.

Registros sem municipio informado, ou com formato invalido, sao direcionados
para um unico membro padrao `NI`, sem nenhuma inferencia de localizacao.

A dimensao descreve a localizacao de **residencia** presente no registro do
beneficio. Ela nao representa necessariamente local de trabalho, endereco do
empregador ou local onde ocorreu um acidente.

In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - FUNCAO DE NORMALIZACAO
# ============================================================
# A funcao e definida uma unica vez e reutilizada tanto na construcao da
# dimensao quanto na preparacao das chaves da fato, evitando duplicar a
# transformacao.

PADRAO_MUNICIPIO_RESIDENCIA = r"^(\d+)-([A-Za-z]{2})-(.+)$"


def normalizar_geografia(df):
    return (
        df
        .withColumn(
            "mun_cod_extraido",
            regexp_extract(col("mun_resid"), PADRAO_MUNICIPIO_RESIDENCIA, 1)
        )
        .withColumn(
            "uf_sigla_extraida",
            regexp_extract(col("mun_resid"), PADRAO_MUNICIPIO_RESIDENCIA, 2)
        )
        .withColumn(
            "municipio_nome_extraido",
            regexp_extract(col("mun_resid"), PADRAO_MUNICIPIO_RESIDENCIA, 3)
        )
        .withColumn(
            "formato_municipio_valido",
            when(col("mun_resid").isNull(), lit(0))
            .when(col("mun_cod_extraido") == "", lit(0))
            .otherwise(lit(1))
        )
        .withColumn(
            "status_geografia_registro",
            when(col("mun_resid").isNull(), lit("nao_informada"))
            .when(col("formato_municipio_valido") == 0, lit("formato_invalido"))
            .otherwise(lit("informada"))
        )
        .withColumn(
            "mun_cod_modelo",
            when(col("status_geografia_registro") == "informada", col("mun_cod"))
            .otherwise(lit("NI"))
        )
        .withColumn(
            "uf_sigla_modelo",
            when(col("status_geografia_registro") == "informada", col("uf_sigla_extraida"))
            .otherwise(lit("NI"))
        )
        .withColumn(
            "mun_nome_modelo",
            when(
                col("status_geografia_registro") == "informada",
                trim(col("municipio_nome_extraido"))
            )
            .otherwise(lit("Localizacao nao identificada"))
        )
        .withColumn(
            "uf_nome_modelo",
            when(
                col("status_geografia_registro") == "informada",
                when(col("uf_sigla_modelo") == "AC", lit("Acre"))
                .when(col("uf_sigla_modelo") == "AL", lit("Alagoas"))
                .when(col("uf_sigla_modelo") == "AP", lit("Amapá"))
                .when(col("uf_sigla_modelo") == "AM", lit("Amazonas"))
                .when(col("uf_sigla_modelo") == "BA", lit("Bahia"))
                .when(col("uf_sigla_modelo") == "CE", lit("Ceará"))
                .when(col("uf_sigla_modelo") == "DF", lit("Distrito Federal"))
                .when(col("uf_sigla_modelo") == "ES", lit("Espírito Santo"))
                .when(col("uf_sigla_modelo") == "GO", lit("Goiás"))
                .when(col("uf_sigla_modelo") == "MA", lit("Maranhão"))
                .when(col("uf_sigla_modelo") == "MT", lit("Mato Grosso"))
                .when(col("uf_sigla_modelo") == "MS", lit("Mato Grosso do Sul"))
                .when(col("uf_sigla_modelo") == "MG", lit("Minas Gerais"))
                .when(col("uf_sigla_modelo") == "PA", lit("Pará"))
                .when(col("uf_sigla_modelo") == "PB", lit("Paraíba"))
                .when(col("uf_sigla_modelo") == "PR", lit("Paraná"))
                .when(col("uf_sigla_modelo") == "PE", lit("Pernambuco"))
                .when(col("uf_sigla_modelo") == "PI", lit("Piauí"))
                .when(col("uf_sigla_modelo") == "RJ", lit("Rio de Janeiro"))
                .when(col("uf_sigla_modelo") == "RN", lit("Rio Grande do Norte"))
                .when(col("uf_sigla_modelo") == "RS", lit("Rio Grande do Sul"))
                .when(col("uf_sigla_modelo") == "RO", lit("Rondônia"))
                .when(col("uf_sigla_modelo") == "RR", lit("Roraima"))
                .when(col("uf_sigla_modelo") == "SC", lit("Santa Catarina"))
                .when(col("uf_sigla_modelo") == "SP", lit("São Paulo"))
                .when(col("uf_sigla_modelo") == "SE", lit("Sergipe"))
                .when(col("uf_sigla_modelo") == "TO", lit("Tocantins"))
                .otherwise(lit("UF nao identificada"))
            )
            .otherwise(lit("Localizacao nao identificada"))
        )
        .withColumn(
            "municipio_origem_modelo",
            when(col("status_geografia_registro") == "informada", col("mun_resid"))
            .otherwise(lit(None))
        )
        .withColumn(
            "status_geografia_dimensao",
            when(col("status_geografia_registro") == "informada", lit("informada"))
            .otherwise(lit("nao_identificada"))
        )
        .withColumn(
            "geografia_sk",
            sha2(concat_ws("||", col("mun_cod_modelo"), col("uf_sigla_modelo")), 256)
        )
    )


In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - DIAGNOSTICO
# ============================================================

df_geografia_diagnostico = normalizar_geografia(
    df_prep.select("mun_resid", "mun_cod", "mun_nome", "uf")
)

qtd_mun_resid_nulo = (
    df_geografia_diagnostico.filter(col("mun_resid").isNull()).count()
)

qtd_formato_invalido = (
    df_geografia_diagnostico
    .filter(col("mun_resid").isNotNull() & (col("formato_municipio_valido") == 0))
    .count()
)

qtd_formato_valido = (
    df_geografia_diagnostico.filter(col("formato_municipio_valido") == 1).count()
)

print("=" * 70)
print("DIAGNOSTICO GEOGRAFICO")
print("=" * 70)
print(f"{'Registros com mun_resid nulo':45}{qtd_mun_resid_nulo:>12,}")
print(f"{'Registros com formato invalido':45}{qtd_formato_invalido:>12,}")
print(f"{'Registros com formato valido':45}{qtd_formato_valido:>12,}")
print("=" * 70)

display(
    df_geografia_diagnostico
    .filter(col("mun_resid").isNotNull() & (col("formato_municipio_valido") == 0))
    .groupBy("mun_resid", "mun_cod", "mun_nome", "uf")
    .count()
    .orderBy("count", ascending=False)
)


### Achado de qualidade geografica

Foram identificados 8 registros cujo campo de municipio nao segue o padrao
esperado `codigo-UF-municipio`.

O valor observado foi `16224-rco do Piaui`, associado a quatro unidades
federativas diferentes na coluna `uf`:

| UF informada | Registros |
|---|---:|
| Piaui | 4 |
| Distrito Federal | 2 |
| Pernambuco | 1 |
| Bahia | 1 |

O trecho geografico nao contem uma sigla de UF valida e o nome do municipio
aparenta estar incompleto. Como a propria coluna `uf` apresenta valores
conflitantes, nao existe evidencia suficiente para reconstruir
automaticamente a localizacao correta.

Decisao adotada:

- preservar o valor original para rastreabilidade (quando informado);
- nao inferir municipio ou UF;
- classificar os registros como `formato_invalido`;
- direcionar os registros para o membro geografico padrao `NI`;
- excluir esses registros de analises municipais ou estaduais que exijam
  localizacao valida.

A regra evita atribuicoes geograficas incorretas e mantem o problema
disponivel para auditoria por meio de `status_geografia_registro`,
preservado na tabela fato.

In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - NORMALIZACAO
# ============================================================
# As colunas de normalizacao ja foram calculadas por normalizar_geografia
# na etapa de diagnostico. Esta etapa apenas confirma o resultado da
# normalizacao antes da criacao da dimensao.

display(
    df_geografia_diagnostico
    .groupBy("status_geografia_registro", "status_geografia_dimensao")
    .count()
    .orderBy("count", ascending=False)
)

display(
    df_geografia_diagnostico
    .filter(col("status_geografia_dimensao") == "nao_identificada")
    .groupBy(
        "mun_cod_modelo",
        "uf_sigla_modelo",
        "mun_nome_modelo",
        "uf_nome_modelo",
        "municipio_origem_modelo"
    )
    .count()
)


In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - CRIACAO
# ============================================================

df_dim_geografia = (
    df_geografia_diagnostico
    .select(
        "geografia_sk",
        col("mun_cod_modelo").alias("mun_cod"),
        col("mun_nome_modelo").alias("mun_nome"),
        col("uf_sigla_modelo").alias("uf_sigla"),
        col("uf_nome_modelo").alias("uf_nome"),
        col("municipio_origem_modelo").alias("municipio_origem"),
        col("status_geografia_dimensao").alias("status_geografia")
    )
    .distinct()
    .withColumn("_data_processamento", current_timestamp())
)

display(df_dim_geografia.orderBy("mun_nome"))


In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - VALIDACAO DA DIMENSAO
# ============================================================

total_dim_geografia = df_dim_geografia.count()

chaves_geografia_distintas = (
    df_dim_geografia.select("geografia_sk").distinct().count()
)

combinacoes_geografia_distintas = (
    df_dim_geografia.select("mun_cod", "uf_sigla").distinct().count()
)

chaves_geografia_nulas = (
    df_dim_geografia.filter(col("geografia_sk").isNull()).count()
)

membros_geografia_ni = (
    df_dim_geografia
    .filter(
        (col("mun_cod") == "NI")
        & (col("uf_sigla") == "NI")
        & (col("status_geografia") == "nao_identificada")
    )
    .count()
)

print("=" * 70)
print("VALIDACAO DA DIMENSAO GEOGRAFICA")
print("=" * 70)
print(f"{'Registros da dimensao':45}{total_dim_geografia:>12,}")
print(f"{'Chaves distintas':45}{chaves_geografia_distintas:>12,}")
print(f"{'Combinacoes mun_cod/uf_sigla distintas':45}{combinacoes_geografia_distintas:>12,}")
print(f"{'Chaves nulas':45}{chaves_geografia_nulas:>12,}")
print(f"{'Membros padrao NI':45}{membros_geografia_ni:>12,}")
print("-" * 70)

erros_dim_geografia = []

if total_dim_geografia != chaves_geografia_distintas:
    erros_dim_geografia.append("A chave geografia_sk nao e unica.")

if total_dim_geografia != combinacoes_geografia_distintas:
    erros_dim_geografia.append(
        "Existe mais de uma linha para a mesma combinacao mun_cod/uf_sigla."
    )

if chaves_geografia_nulas > 0:
    erros_dim_geografia.append("Existem chaves geografia_sk nulas.")

if membros_geografia_ni != 1:
    erros_dim_geografia.append(
        "A dimensao deve possuir exatamente um membro padrao NI."
    )

if not erros_dim_geografia:
    print("RESULTADO: OK")
    print(
        "A dimensao possui uma linha por combinacao geografica e um "
        "unico membro padrao para localizacao nao identificada."
    )
else:
    print("RESULTADO: FALHA")
    for erro in erros_dim_geografia:
        print(f"- {erro}")

print("=" * 70)

if erros_dim_geografia:
    raise ValueError(
        "Validacao da dimensao geografica falhou: "
        + " | ".join(erros_dim_geografia)
    )


In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - VALIDACAO DE COBERTURA
# ============================================================
# A dimensao descreve municipio de residencia, nao local de trabalho.

df_teste_cobertura_geografia = (
    df_geografia_diagnostico.alias("f")
    .join(
        df_dim_geografia.alias("d"),
        col("f.geografia_sk") == col("d.geografia_sk"),
        "left"
    )
)

registros_apos_join_geografia = df_teste_cobertura_geografia.count()

registros_sem_geografia = (
    df_teste_cobertura_geografia.filter(col("d.geografia_sk").isNull()).count()
)

print("=" * 70)
print("VALIDACAO DE COBERTURA DA DIMENSAO GEOGRAFICA")
print("=" * 70)
print(f"{'Registros na origem':45}{quantidade_linhas_prep:>12,}")
print(f"{'Registros apos relacionamento':45}{registros_apos_join_geografia:>12,}")
print(f"{'Registros sem correspondencia':45}{registros_sem_geografia:>12,}")
print("-" * 70)

erros_cobertura_geografia = []

if registros_apos_join_geografia != quantidade_linhas_prep:
    erros_cobertura_geografia.append(
        "O relacionamento multiplicou ou removeu registros."
    )

if registros_sem_geografia > 0:
    erros_cobertura_geografia.append("Existem registros sem correspondencia.")

if not erros_cobertura_geografia:
    print("RESULTADO: OK")
    print("Todos os registros encontram correspondencia na dimensao geografica.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_cobertura_geografia:
        print(f"- {erro}")

print("=" * 70)

if erros_cobertura_geografia:
    raise ValueError(
        "Validacao de cobertura da dimensao geografica falhou: "
        + " | ".join(erros_cobertura_geografia)
    )


In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - GRAVACAO
# ============================================================

(
    df_dim_geografia
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_GEOGRAFIA)
)

print(f"Tabela gravada: {TABELA_DIM_GEOGRAFIA}")


In [0]:
# ============================================================
# DIMENSAO GEOGRAFICA - VALIDACAO DA PERSISTENCIA
# ============================================================

df_dim_geografia_persistida = spark.table(TABELA_DIM_GEOGRAFIA)

linhas_geografia_antes = df_dim_geografia.count()
linhas_geografia_depois = df_dim_geografia_persistida.count()

print("=" * 70)
print("VALIDACAO DA PERSISTENCIA DA DIMENSAO GEOGRAFICA")
print("=" * 70)
print(f"{'Antes da gravacao':40}{linhas_geografia_antes:>12,}")
print(f"{'Depois da gravacao':40}{linhas_geografia_depois:>12,}")
print("-" * 70)

if linhas_geografia_antes == linhas_geografia_depois:
    print("RESULTADO: OK")
    print("A dimensao geografica foi persistida sem alteracao de registros.")
else:
    print("RESULTADO: FALHA")
    print("A persistencia alterou a quantidade de registros.")

print("=" * 70)

if linhas_geografia_antes != linhas_geografia_depois:
    raise ValueError(
        "Persistencia da dimensao geografica alterou a quantidade "
        "de registros."
    )


In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.dim_geografia
IS 'Dimensao geografica de residencia utilizada no modelo analitico de beneficios e afastamentos. Possui uma linha por combinacao de codigo de municipio e sigla de UF. Representa municipio de residencia, nao local de trabalho.';

ALTER TABLE afastamento_inss.gold.dim_geografia
ALTER COLUMN geografia_sk
COMMENT 'Chave tecnica deterministica da geografia, gerada por SHA-256 sobre mun_cod e uf_sigla normalizados.';

ALTER TABLE afastamento_inss.gold.dim_geografia
ALTER COLUMN mun_cod
COMMENT 'Codigo do municipio de residencia ou NI quando nao identificado.';

ALTER TABLE afastamento_inss.gold.dim_geografia
ALTER COLUMN mun_nome
COMMENT 'Nome do municipio de residencia extraido de mun_resid ou classificacao tecnica de ausencia.';

ALTER TABLE afastamento_inss.gold.dim_geografia
ALTER COLUMN uf_sigla
COMMENT 'Sigla da UF de residencia extraida de mun_resid ou NI quando nao identificada.';

ALTER TABLE afastamento_inss.gold.dim_geografia
ALTER COLUMN municipio_origem
COMMENT 'Valor original do campo mun_resid, preservado para rastreabilidade quando a geografia foi informada.';

ALTER TABLE afastamento_inss.gold.dim_geografia
ALTER COLUMN status_geografia
COMMENT 'Status da geografia na dimensao: informada ou nao_identificada.';


## 6. Dimensao de atividade

Grao da dimensao:

> Uma linha por combinacao distinta de `ramo_atividade`, `cnae_2023` e
> `cnae_2024`.

CNAE 2023 e CNAE 2024 nao sao simplificados em um unico codigo sem regra
explicita: ambos sao preservados na dimensao, e um campo adicional
`cnae_referencia` indica qual dos dois foi usado como referencia analitica,
priorizando CNAE 2024 e utilizando CNAE 2023 apenas na ausencia do primeiro.

O CNAE representa a atividade economica quando informada pela fonte. A
ausencia de CNAE nao significa que a pessoa nao trabalha; ela apenas indica
que a fonte nao disponibilizou essa informacao para o registro.

In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - DIAGNOSTICO DE COBERTURA DE CNAE
# ============================================================

qtd_cnae_2023_informado = df_prep.filter(col("cnae_2023").isNotNull()).count()
qtd_cnae_2024_informado = df_prep.filter(col("cnae_2024").isNotNull()).count()
qtd_ramo_atividade_nulo = df_prep.filter(col("ramo_atividade").isNull()).count()

pct_cnae_2023 = 100 * qtd_cnae_2023_informado / quantidade_linhas_prep
pct_cnae_2024 = 100 * qtd_cnae_2024_informado / quantidade_linhas_prep

print("=" * 70)
print("DIAGNOSTICO DE COBERTURA DE CNAE")
print("=" * 70)
print(f"{'Registros com CNAE 2023 informado':45}{qtd_cnae_2023_informado:>12,} ({pct_cnae_2023:.2f}%)")
print(f"{'Registros com CNAE 2024 informado':45}{qtd_cnae_2024_informado:>12,} ({pct_cnae_2024:.2f}%)")
print(f"{'Registros com ramo_atividade nulo':45}{qtd_ramo_atividade_nulo:>12,}")
print("=" * 70)


In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - FUNCAO DE NORMALIZACAO
# ============================================================
# A funcao e definida uma unica vez e reutilizada tanto na construcao da
# dimensao quanto na preparacao das chaves da fato.

def normalizar_atividade(df):
    return (
        df
        .withColumn(
            "cnae_2023_modelo",
            when(col("cnae_2023").isNull(), lit("NI")).otherwise(col("cnae_2023"))
        )
        .withColumn(
            "cnae_2024_modelo",
            when(col("cnae_2024").isNull(), lit("NI")).otherwise(col("cnae_2024"))
        )
        .withColumn(
            "cnae_referencia",
            when(col("cnae_2024").isNotNull(), col("cnae_2024"))
            .when(col("cnae_2023").isNotNull(), col("cnae_2023"))
            .otherwise(lit("NI"))
        )
        .withColumn(
            "cnae_origem",
            when(col("cnae_2024").isNotNull(), lit("CNAE 2024"))
            .when(col("cnae_2023").isNotNull(), lit("CNAE 2023"))
            .otherwise(lit("Nao informado"))
        )
        .withColumn(
            "status_cnae",
            when(col("cnae_referencia") == "NI", lit("nao_informado"))
            .otherwise(lit("informado"))
        )
        .withColumn(
            "ind_cnae_informado",
            when(col("cnae_referencia") == "NI", lit(0)).otherwise(lit(1))
        )
        .withColumn(
            "atividade_sk",
            sha2(
                concat_ws(
                    "||",
                    col("ramo_atividade"),
                    col("cnae_2023_modelo"),
                    col("cnae_2024_modelo")
                ),
                256
            )
        )
    )


df_atividade_normalizada = normalizar_atividade(
    df_prep.select("ramo_atividade", "cnae_2023", "cnae_2024")
)

display(
    df_atividade_normalizada
    .groupBy("cnae_origem", "status_cnae")
    .count()
    .orderBy("count", ascending=False)
)


In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - CRIACAO
# ============================================================

df_dim_atividade = (
    df_atividade_normalizada
    .select(
        "atividade_sk",
        "ramo_atividade",
        "cnae_2023",
        "cnae_2024",
        "cnae_referencia",
        "cnae_origem",
        "status_cnae",
        "ind_cnae_informado"
    )
    .distinct()
    .withColumn("_data_processamento", current_timestamp())
)

display(df_dim_atividade.orderBy("status_cnae", "ramo_atividade"))


In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - VALIDACAO
# ============================================================

total_dim_atividade = df_dim_atividade.count()

chaves_atividade_distintas = (
    df_dim_atividade.select("atividade_sk").distinct().count()
)

combinacoes_atividade_distintas = (
    df_dim_atividade
    .select("ramo_atividade", "cnae_2023", "cnae_2024")
    .distinct()
    .count()
)

chaves_atividade_nulas = (
    df_dim_atividade.filter(col("atividade_sk").isNull()).count()
)

print("=" * 70)
print("VALIDACAO DA DIMENSAO DE ATIVIDADE")
print("=" * 70)
print(f"{'Registros da dimensao':45}{total_dim_atividade:>12,}")
print(f"{'Chaves distintas':45}{chaves_atividade_distintas:>12,}")
print(f"{'Combinacoes ramo/cnae distintas':45}{combinacoes_atividade_distintas:>12,}")
print(f"{'Chaves nulas':45}{chaves_atividade_nulas:>12,}")
print("-" * 70)

erros_dim_atividade = []

if total_dim_atividade != chaves_atividade_distintas:
    erros_dim_atividade.append("A chave atividade_sk nao e unica.")

if total_dim_atividade != combinacoes_atividade_distintas:
    erros_dim_atividade.append(
        "Existe mais de uma linha para a mesma combinacao de atividade."
    )

if chaves_atividade_nulas > 0:
    erros_dim_atividade.append("Existem chaves atividade_sk nulas.")

if not erros_dim_atividade:
    print("RESULTADO: OK")
    print(
        "A dimensao possui uma linha por combinacao distinta de "
        "ramo_atividade, cnae_2023 e cnae_2024."
    )
else:
    print("RESULTADO: FALHA")
    for erro in erros_dim_atividade:
        print(f"- {erro}")

print("=" * 70)

if erros_dim_atividade:
    raise ValueError(
        "Validacao da dimensao de atividade falhou: "
        + " | ".join(erros_dim_atividade)
    )


In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - VALIDACAO DE COBERTURA
# ============================================================

df_teste_cobertura_atividade = (
    df_atividade_normalizada.alias("f")
    .join(
        df_dim_atividade.alias("d"),
        col("f.atividade_sk") == col("d.atividade_sk"),
        "left"
    )
)

registros_apos_join_atividade = df_teste_cobertura_atividade.count()

registros_sem_atividade = (
    df_teste_cobertura_atividade.filter(col("d.atividade_sk").isNull()).count()
)

print("=" * 70)
print("VALIDACAO DE COBERTURA DA DIMENSAO DE ATIVIDADE")
print("=" * 70)
print(f"{'Registros na origem':45}{quantidade_linhas_prep:>12,}")
print(f"{'Registros apos relacionamento':45}{registros_apos_join_atividade:>12,}")
print(f"{'Registros sem correspondencia':45}{registros_sem_atividade:>12,}")
print("-" * 70)

erros_cobertura_atividade = []

if registros_apos_join_atividade != quantidade_linhas_prep:
    erros_cobertura_atividade.append(
        "O relacionamento multiplicou ou removeu registros."
    )

if registros_sem_atividade > 0:
    erros_cobertura_atividade.append("Existem registros sem correspondencia.")

if not erros_cobertura_atividade:
    print("RESULTADO: OK")
    print("Todos os registros encontram correspondencia na dimensao de atividade.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_cobertura_atividade:
        print(f"- {erro}")

print("=" * 70)

if erros_cobertura_atividade:
    raise ValueError(
        "Validacao de cobertura da dimensao de atividade falhou: "
        + " | ".join(erros_cobertura_atividade)
    )


In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - GRAVACAO
# ============================================================

(
    df_dim_atividade
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_ATIVIDADE)
)

print(f"Tabela gravada: {TABELA_DIM_ATIVIDADE}")


In [0]:
# ============================================================
# DIMENSAO DE ATIVIDADE - VALIDACAO DA PERSISTENCIA
# ============================================================

df_dim_atividade_persistida = spark.table(TABELA_DIM_ATIVIDADE)

linhas_atividade_antes = df_dim_atividade.count()
linhas_atividade_depois = df_dim_atividade_persistida.count()

print("=" * 70)
print("VALIDACAO DA PERSISTENCIA DA DIMENSAO DE ATIVIDADE")
print("=" * 70)
print(f"{'Antes da gravacao':40}{linhas_atividade_antes:>12,}")
print(f"{'Depois da gravacao':40}{linhas_atividade_depois:>12,}")
print("-" * 70)

if linhas_atividade_antes == linhas_atividade_depois:
    print("RESULTADO: OK")
    print("A dimensao de atividade foi persistida sem alteracao de registros.")
else:
    print("RESULTADO: FALHA")
    print("A persistencia alterou a quantidade de registros.")

print("=" * 70)

if linhas_atividade_antes != linhas_atividade_depois:
    raise ValueError(
        "Persistencia da dimensao de atividade alterou a quantidade "
        "de registros."
    )


In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.dim_atividade
IS 'Dimensao de atividade economica utilizada no modelo analitico de beneficios e afastamentos. Possui uma linha por combinacao distinta de ramo_atividade, cnae_2023 e cnae_2024. A ausencia de CNAE nao significa que a pessoa nao trabalha.';

ALTER TABLE afastamento_inss.gold.dim_atividade
ALTER COLUMN atividade_sk
COMMENT 'Chave tecnica deterministica da atividade, gerada por SHA-256 sobre ramo_atividade, cnae_2023 e cnae_2024.';

ALTER TABLE afastamento_inss.gold.dim_atividade
ALTER COLUMN cnae_referencia
COMMENT 'Codigo CNAE de referencia analitica: prioriza CNAE 2024 e utiliza CNAE 2023 apenas na ausencia do primeiro. NI quando nenhum dos dois foi informado.';

ALTER TABLE afastamento_inss.gold.dim_atividade
ALTER COLUMN cnae_origem
COMMENT 'Indica qual versao do CNAE foi utilizada como referencia: CNAE 2024, CNAE 2023 ou Nao informado.';

ALTER TABLE afastamento_inss.gold.dim_atividade
ALTER COLUMN status_cnae
COMMENT 'Status de preenchimento do CNAE de referencia: informado ou nao_informado.';


## 7. Fato de afastamentos

Grao da fato:

> Uma linha por registro da Gold preparatoria classificado como afastamento
> (`ind_afastamento = 1`).

A fato relaciona as cinco dimensoes por meio de chaves tecnicas
deterministicas, calculadas com a mesma logica utilizada na construcao de
cada dimensao (funcoes `normalizar_geografia` e `normalizar_atividade`
reaproveitadas, e derivacao identica de `cid_cod_modelo`), garantindo
integridade referencial total.

O modelo definido para este notebook contempla apenas as cinco dimensoes
solicitadas (tempo, CID, especie, geografia e atividade). Atributos do
registro que nao pertencem a nenhuma delas — por exemplo dados
administrativos e demograficos do beneficio (`sexo`, `clientela`,
`forma_filiacao`, `grau_instrucao`, `faixa_etaria`, `idade_na_competencia`,
`qt_anos_contribuicao`, `qt_sm_rmi`, `dt_nascimento`, `dt_dib`, `dt_ddb`,
`dt_dcb`) — sao preservados como atributos degenerados diretamente na fato,
em vez de terem uma dimensao inventada para eles, o que nao foi solicitado
na especificacao do modelo.

A fato nao representa pessoas, beneficiarios ou trabalhadores unicos: cada
linha e um registro de beneficio classificado como afastamento, podendo o
mesmo individuo (nao identificavel na fonte) estar associado a mais de um
registro.

In [0]:
# ============================================================
# FATO - FILTRO DO GRAO E PREPARACAO DAS CHAVES DIMENSIONAIS
# ============================================================

df_fato_base = df_prep.filter(col("ind_afastamento") == 1)

quantidade_afastamentos = df_fato_base.count()
percentual_afastamentos = 100 * quantidade_afastamentos / quantidade_linhas_prep

print("=" * 70)
print("FILTRO DO GRAO DA FATO")
print("=" * 70)
print(f"{'Registros na Gold preparatoria':45}{quantidade_linhas_prep:>12,}")
print(f"{'Registros classificados como afastamento':45}{quantidade_afastamentos:>12,} ({percentual_afastamentos:.2f}%)")
print("=" * 70)

df_fato_chaves = (
    df_fato_base
    .withColumn(
        "cid_cod_modelo",
        when(col("cid_cod").isNull(), lit("NI")).otherwise(col("cid_cod"))
    )
    .withColumn("tempo_sk", sha2(col("competencia_concessao"), 256))
    .withColumn("cid_sk", sha2(col("cid_cod_modelo"), 256))
    .withColumn("especie_sk", sha2(col("especie_cod"), 256))
)

df_fato_chaves = normalizar_atividade(df_fato_chaves)
df_fato_chaves = normalizar_geografia(df_fato_chaves)

chaves_fato_nulas = (
    df_fato_chaves
    .filter(
        col("tempo_sk").isNull()
        | col("cid_sk").isNull()
        | col("especie_sk").isNull()
        | col("geografia_sk").isNull()
        | col("atividade_sk").isNull()
    )
    .count()
)

print(f"Registros com alguma chave dimensional nula: {chaves_fato_nulas:,}")

if chaves_fato_nulas > 0:
    raise ValueError(
        "Existem registros com chave dimensional nula na preparacao da fato."
    )


In [0]:
# ============================================================
# FATO - CRIACAO
# Dependencias na celula centralizada de imports:
# coalesce, row_number e Window
# ============================================================

df_fato_base = (
    df_fato_chaves
    .select(
        "tempo_sk",
        "cid_sk",
        "especie_sk",
        "geografia_sk",
        "atividade_sk",
        "status_geografia_registro",
        "dt_dib",
        "dt_ddb",
        "dt_dcb",
        "duracao_beneficio_dias",
        "qtd_beneficios",
        "ind_cid_informado",
        "ind_saude_mental",
        "ind_osteomuscular",
        "ind_cardiovascular",
        "ind_respiratorio",
        "ind_acidentario",
        "ind_afastamento_saude_mental",
        "ind_afastamento_osteomuscular",
        "ind_afastamento_cardiovascular",
        "ind_afastamento_respiratorio",
        "dt_nascimento",
        "idade_na_competencia",
        "faixa_etaria",
        "sexo",
        "clientela",
        "forma_filiacao",
        "grau_instrucao",
        "qt_anos_contribuicao",
        "qt_sm_rmi"
    )
)

COLUNAS_EXCLUIDAS_HASH = {
    "fato_sk",
    "fato_hash_base",
    "registro_ocorrencia",
    "_data_processamento"
}

colunas_hash_fato = sorted([
    nome_coluna
    for nome_coluna in df_fato_base.columns
    if nome_coluna not in COLUNAS_EXCLUIDAS_HASH
])

if not colunas_hash_fato:
    raise ValueError(
        "Nenhuma coluna disponivel para gerar a chave tecnica da fato."
    )

df_fato_com_hash = (
    df_fato_base
    .withColumn(
        "fato_hash_base",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(nome_coluna).cast("string"),
                        lit("<NULL>")
                    )
                    for nome_coluna in colunas_hash_fato
                ]
            ),
            256
        )
    )
)

janela_ocorrencia_fato = (
    Window
    .partitionBy("fato_hash_base")
    .orderBy(
        *[
            col(nome_coluna).asc_nulls_last()
            for nome_coluna in colunas_hash_fato
        ]
    )
)

df_fato_afastamentos = (
    df_fato_com_hash
    .withColumn(
        "registro_ocorrencia",
        row_number().over(janela_ocorrencia_fato)
    )
    .withColumn(
        "fato_sk",
        sha2(
            concat_ws(
                "||",
                col("fato_hash_base"),
                col("registro_ocorrencia").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "_data_processamento",
        current_timestamp()
    )
)

colunas_prioritarias_fato = [
    "fato_sk",
    "fato_hash_base",
    "registro_ocorrencia",
    "tempo_sk",
    "cid_sk",
    "especie_sk",
    "geografia_sk",
    "atividade_sk"
]

colunas_restantes_fato = [
    nome_coluna
    for nome_coluna in df_fato_afastamentos.columns
    if nome_coluna not in colunas_prioritarias_fato
]

df_fato_afastamentos = df_fato_afastamentos.select(
    *colunas_prioritarias_fato,
    *colunas_restantes_fato
)

total_registros_fato = df_fato_afastamentos.count()

total_fato_sk_distintas = (
    df_fato_afastamentos
    .select("fato_sk")
    .distinct()
    .count()
)

total_fato_sk_nulas = (
    df_fato_afastamentos
    .filter(col("fato_sk").isNull())
    .count()
)

total_hash_base_nulos = (
    df_fato_afastamentos
    .filter(col("fato_hash_base").isNull())
    .count()
)

ocorrencias_invalidas = (
    df_fato_afastamentos
    .filter(col("registro_ocorrencia") < 1)
    .count()
)

qtd_beneficios_fato = (
    df_fato_afastamentos
    .agg(
        spark_sum("qtd_beneficios").alias("total")
    )
    .first()["total"]
)

print("=" * 75)
print("VALIDACAO DA CHAVE TECNICA DA FATO")
print("=" * 75)
print(
    f"{'Total de registros':50}"
    f"{total_registros_fato:>15,}"
)
print(
    f"{'fato_sk distintas':50}"
    f"{total_fato_sk_distintas:>15,}"
)
print(
    f"{'fato_sk nulas':50}"
    f"{total_fato_sk_nulas:>15,}"
)
print(
    f"{'fato_hash_base nulos':50}"
    f"{total_hash_base_nulos:>15,}"
)
print(
    f"{'Ocorrencias invalidas':50}"
    f"{ocorrencias_invalidas:>15,}"
)
print(
    f"{'Soma de qtd_beneficios':50}"
    f"{qtd_beneficios_fato:>15,}"
)
print("-" * 75)

erros_chave_fato = []

if total_registros_fato != total_fato_sk_distintas:
    erros_chave_fato.append(
        "A fato_sk nao e unica."
    )

if total_fato_sk_nulas > 0:
    erros_chave_fato.append(
        "Existem valores nulos em fato_sk."
    )

if total_hash_base_nulos > 0:
    erros_chave_fato.append(
        "Existem valores nulos em fato_hash_base."
    )

if ocorrencias_invalidas > 0:
    erros_chave_fato.append(
        "Existem ocorrencias com valor inferior a 1."
    )

if qtd_beneficios_fato != total_registros_fato:
    erros_chave_fato.append(
        "A soma de qtd_beneficios difere do total de registros."
    )

if erros_chave_fato:
    raise ValueError(
        "Falha na chave tecnica da fato: "
        + " | ".join(erros_chave_fato)
    )

print("RESULTADO: OK")
print(
    "A fato possui uma chave tecnica unica e nao nula "
    "para cada registro."
)
print("=" * 75)

display(df_fato_afastamentos)

In [0]:
# ============================================================
# FATO - VALIDACOES
# ============================================================

total_fato = df_fato_afastamentos.count()

chaves_nulas_fato = (
    df_fato_afastamentos
    .filter(
        col("tempo_sk").isNull()
        | col("cid_sk").isNull()
        | col("especie_sk").isNull()
        | col("geografia_sk").isNull()
        | col("atividade_sk").isNull()
    )
    .count()
)

soma_qtd_beneficios_fato = (
    df_fato_afastamentos.agg(spark_sum(col("qtd_beneficios"))).collect()[0][0]
)

soma_qtd_beneficios_origem = (
    df_fato_base.agg(spark_sum(col("qtd_beneficios"))).collect()[0][0]
)

print("=" * 70)
print("VALIDACOES DA FATO")
print("=" * 70)
print(f"{'Registros da fato':45}{total_fato:>12,}")
print(f"{'Registros classificados como afastamento na origem':45}{quantidade_afastamentos:>12,}")
print(f"{'Registros com chave dimensional nula':45}{chaves_nulas_fato:>12,}")
print(f"{'Soma qtd_beneficios na fato':45}{soma_qtd_beneficios_fato:>12,}")
print(f"{'Soma qtd_beneficios na origem filtrada':45}{soma_qtd_beneficios_origem:>12,}")
print("-" * 70)

erros_fato = []

if total_fato != quantidade_afastamentos:
    erros_fato.append(
        "A quantidade de linhas da fato diverge dos registros classificados "
        "como afastamento na origem."
    )

if chaves_nulas_fato > 0:
    erros_fato.append("Existem registros com chave dimensional nula na fato.")

if soma_qtd_beneficios_fato != soma_qtd_beneficios_origem:
    erros_fato.append(
        "A soma de qtd_beneficios da fato diverge da soma na origem filtrada."
    )

if not erros_fato:
    print("RESULTADO: OK")
    print(
        "A fato possui uma linha por registro classificado como afastamento, "
        "sem chaves nulas e sem perda de medidas."
    )
else:
    print("RESULTADO: FALHA")
    for erro in erros_fato:
        print(f"- {erro}")

print("=" * 70)

if erros_fato:
    raise ValueError(
        "Validacao da fato de afastamentos falhou: " + " | ".join(erros_fato)
    )


In [0]:
# ============================================================
# FATO - VALIDACAO DE INTEGRIDADE REFERENCIAL
# ============================================================
# Cada chave dimensional da fato e confrontada com a respectiva dimensao
# ja persistida, garantindo que todo registro encontre correspondencia e
# que o relacionamento nao multiplique nem remova linhas.

erros_integridade_referencial = []

df_orfaos_tempo = (
    df_fato_afastamentos
    .join(df_dim_tempo_persistida.select("tempo_sk"), "tempo_sk", "left_anti")
)
qtd_orfaos_tempo = df_orfaos_tempo.count()

df_orfaos_cid = (
    df_fato_afastamentos
    .join(df_dim_cid_persistida.select("cid_sk"), "cid_sk", "left_anti")
)
qtd_orfaos_cid = df_orfaos_cid.count()

df_orfaos_especie = (
    df_fato_afastamentos
    .join(df_dim_especie_persistida.select("especie_sk"), "especie_sk", "left_anti")
)
qtd_orfaos_especie = df_orfaos_especie.count()

df_orfaos_geografia = (
    df_fato_afastamentos
    .join(df_dim_geografia_persistida.select("geografia_sk"), "geografia_sk", "left_anti")
)
qtd_orfaos_geografia = df_orfaos_geografia.count()

df_orfaos_atividade = (
    df_fato_afastamentos
    .join(df_dim_atividade_persistida.select("atividade_sk"), "atividade_sk", "left_anti")
)
qtd_orfaos_atividade = df_orfaos_atividade.count()

print("=" * 70)
print("VALIDACAO DE INTEGRIDADE REFERENCIAL DA FATO")
print("=" * 70)
print(f"{'Registros sem correspondencia em dim_tempo':45}{qtd_orfaos_tempo:>12,}")
print(f"{'Registros sem correspondencia em dim_cid':45}{qtd_orfaos_cid:>12,}")
print(f"{'Registros sem correspondencia em dim_especie':45}{qtd_orfaos_especie:>12,}")
print(f"{'Registros sem correspondencia em dim_geografia':45}{qtd_orfaos_geografia:>12,}")
print(f"{'Registros sem correspondencia em dim_atividade':45}{qtd_orfaos_atividade:>12,}")
print("-" * 70)

if qtd_orfaos_tempo > 0:
    erros_integridade_referencial.append("Existem registros sem correspondencia em dim_tempo.")

if qtd_orfaos_cid > 0:
    erros_integridade_referencial.append("Existem registros sem correspondencia em dim_cid.")

if qtd_orfaos_especie > 0:
    erros_integridade_referencial.append("Existem registros sem correspondencia em dim_especie.")

if qtd_orfaos_geografia > 0:
    erros_integridade_referencial.append("Existem registros sem correspondencia em dim_geografia.")

if qtd_orfaos_atividade > 0:
    erros_integridade_referencial.append("Existem registros sem correspondencia em dim_atividade.")

if not erros_integridade_referencial:
    print("RESULTADO: OK")
    print("Todas as chaves dimensionais da fato encontram correspondencia.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_integridade_referencial:
        print(f"- {erro}")

print("=" * 70)

if erros_integridade_referencial:
    raise ValueError(
        "Validacao de integridade referencial da fato falhou: "
        + " | ".join(erros_integridade_referencial)
    )


In [0]:
# ============================================================
# FATO - GRAVACAO
# ============================================================

(
    df_fato_afastamentos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FATO)
)

print(f"Tabela gravada: {TABELA_FATO}")


In [0]:
# ============================================================
# FATO - VALIDACAO DA PERSISTENCIA
# ============================================================

df_fato_persistida = spark.table(TABELA_FATO)

linhas_fato_antes = df_fato_afastamentos.count()
linhas_fato_depois = df_fato_persistida.count()

print("=" * 70)
print("VALIDACAO DA PERSISTENCIA DA FATO")
print("=" * 70)
print(f"{'Antes da gravacao':40}{linhas_fato_antes:>12,}")
print(f"{'Depois da gravacao':40}{linhas_fato_depois:>12,}")
print("-" * 70)

if linhas_fato_antes == linhas_fato_depois:
    print("RESULTADO: OK")
    print("A fato foi persistida sem alteracao de registros.")
else:
    print("RESULTADO: FALHA")
    print("A persistencia alterou a quantidade de registros.")

print("=" * 70)

if linhas_fato_antes != linhas_fato_depois:
    raise ValueError(
        "Persistencia da fato de afastamentos alterou a quantidade "
        "de registros."
    )


In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.fato_afastamentos
IS 'Fato de registros de beneficios do INSS classificados como afastamento. Possui uma linha por registro de afastamento, nao por pessoa ou trabalhador unico, pois a fonte nao possui identificador individual validado.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN tempo_sk
COMMENT 'Chave estrangeira para afastamento_inss.gold.dim_tempo.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN cid_sk
COMMENT 'Chave estrangeira para afastamento_inss.gold.dim_cid.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN especie_sk
COMMENT 'Chave estrangeira para afastamento_inss.gold.dim_especie.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN geografia_sk
COMMENT 'Chave estrangeira para afastamento_inss.gold.dim_geografia. Representa municipio de residencia, nao local de trabalho.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN atividade_sk
COMMENT 'Chave estrangeira para afastamento_inss.gold.dim_atividade. A ausencia de CNAE nao significa que a pessoa nao trabalha.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN status_geografia_registro
COMMENT 'Status geografico do registro antes da normalizacao para a dimensao: informada, nao_informada ou formato_invalido. Preservado para auditoria.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN qtd_beneficios
COMMENT 'Medida aditiva: quantidade de beneficios do registro, conforme a Gold preparatoria.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN duracao_beneficio_dias
COMMENT 'Duracao do beneficio em dias, conforme calculada na Gold preparatoria.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_saude_mental
COMMENT 'Indicador binario: 1 quando o CID pertence ao grupo mental e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_osteomuscular
COMMENT 'Indicador binario: 1 quando o CID pertence ao grupo osteomuscular e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_cardiovascular
COMMENT 'Indicador binario: 1 quando o CID pertence ao grupo cardiovascular e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_respiratorio
COMMENT 'Indicador binario: 1 quando o CID pertence ao grupo respiratorio e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_acidentario
COMMENT 'Indicador binario: 1 para afastamento de natureza acidentaria e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_afastamento_saude_mental
COMMENT 'Indicador composto: 1 quando o afastamento esta associado a transtornos mentais e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_afastamento_osteomuscular
COMMENT 'Indicador composto: 1 quando o afastamento esta associado a doencas osteomusculares e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_afastamento_cardiovascular
COMMENT 'Indicador composto: 1 quando o afastamento esta associado a doencas cardiovasculares e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.fato_afastamentos
ALTER COLUMN ind_afastamento_respiratorio
COMMENT 'Indicador composto: 1 quando o afastamento esta associado a doencas respiratorias e 0 para os demais.';


## 8. Validacao geral do modelo estrela e encerramento

Etapa final: confirma que as cinco dimensoes e a fato estao persistidas e
referencialmente integras, resume as contagens de cada tabela do modelo e
libera o cache do DataFrame de origem.

In [0]:
# ============================================================
# VALIDACAO GERAL DO MODELO ESTRELA
# ============================================================
# Releitura das tabelas persistidas e reavaliacao da integridade
# referencial entre a fato e cada dimensao, confirmando o estado final
# do modelo no Unity Catalog.

df_fato_final = spark.table(TABELA_FATO)

erros_modelo_estrela = []

for nome_tabela, tabela, coluna_sk in [
    ("dim_tempo", TABELA_DIM_TEMPO, "tempo_sk"),
    ("dim_cid", TABELA_DIM_CID, "cid_sk"),
    ("dim_especie", TABELA_DIM_ESPECIE, "especie_sk"),
    ("dim_geografia", TABELA_DIM_GEOGRAFIA, "geografia_sk"),
    ("dim_atividade", TABELA_DIM_ATIVIDADE, "atividade_sk")
]:
    df_dimensao_final = spark.table(tabela)

    qtd_dimensao = df_dimensao_final.count()

    qtd_orfaos = (
        df_fato_final
        .join(df_dimensao_final.select(coluna_sk), coluna_sk, "left_anti")
        .count()
    )

    print(f"{nome_tabela:20}{qtd_dimensao:>12,} registros{'':5}orfaos na fato: {qtd_orfaos:>10,}")

    if qtd_dimensao == 0:
        erros_modelo_estrela.append(f"A dimensao {nome_tabela} esta vazia.")

    if qtd_orfaos > 0:
        erros_modelo_estrela.append(
            f"Existem registros orfaos na fato em relacao a {nome_tabela}."
        )

print("=" * 70)

if not erros_modelo_estrela:
    print("RESULTADO: OK")
    print("O modelo estrela esta completo e referencialmente integro.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_modelo_estrela:
        print(f"- {erro}")

print("=" * 70)

if erros_modelo_estrela:
    raise ValueError(
        "Validacao geral do modelo estrela falhou: "
        + " | ".join(erros_modelo_estrela)
    )


In [0]:
# ============================================================
# RESUMO DE CONTAGENS
# ============================================================

resumo_contagens = [
    ("prep_beneficios_bi (origem)", quantidade_linhas_prep),
    ("dim_tempo", spark.table(TABELA_DIM_TEMPO).count()),
    ("dim_cid", spark.table(TABELA_DIM_CID).count()),
    ("dim_especie", spark.table(TABELA_DIM_ESPECIE).count()),
    ("dim_geografia", spark.table(TABELA_DIM_GEOGRAFIA).count()),
    ("dim_atividade", spark.table(TABELA_DIM_ATIVIDADE).count()),
    ("fato_afastamentos", spark.table(TABELA_FATO).count())
]

print("=" * 70)
print("RESUMO DE CONTAGENS DO MODELO")
print("=" * 70)
for nome_tabela, quantidade in resumo_contagens:
    print(f"{nome_tabela:45}{quantidade:>12,}")
print("-" * 70)
print(
    f"{'Percentual de registros classificados como afastamento':45}"
    f"{percentual_afastamentos:>11.2f}%"
)
print("=" * 70)


In [0]:
# ============================================================
# ENCERRAMENTO E LIBERACAO DE CACHE
# ============================================================

print("Pipeline concluido com sucesso.")


In [0]:
%sql
-- ============================================================
-- 1. VALIDACAO GERAL DA FATO
-- ============================================================

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT fato_sk) AS fato_sk_distintas,
    SUM(
        CASE
            WHEN fato_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS fato_sk_nulas,
    COUNT(DISTINCT fato_hash_base) AS hashes_base_distintos,
    MAX(registro_ocorrencia) AS maior_registro_ocorrencia,
    SUM(qtd_beneficios) AS qtd_beneficios,
    SUM(ind_saude_mental) AS qtd_saude_mental,
    SUM(ind_osteomuscular) AS qtd_osteomuscular,
    SUM(ind_acidentario) AS qtd_acidentario,
    SUM(ind_afastamento_saude_mental)
        AS qtd_afastamento_saude_mental,
    SUM(ind_afastamento_osteomuscular)
        AS qtd_afastamento_osteomuscular,
    SUM(ind_cardiovascular)
        AS qtd_cardiovascular,
    SUM(ind_respiratorio)
        AS qtd_respiratorio,
    SUM(ind_afastamento_cardiovascular)
        AS qtd_afastamento_cardiovascular,
    SUM(ind_afastamento_respiratorio)
        AS qtd_afastamento_respiratorio
FROM afastamento_inss.gold.fato_afastamentos;

In [0]:
%sql
-- ============================================================
-- 2. RESULTADO CONSOLIDADO DA CHAVE TECNICA
-- ============================================================

SELECT
    CASE
        WHEN COUNT(*) = 185412
         AND COUNT(*) = COUNT(DISTINCT fato_sk)
         AND SUM(
                CASE
                    WHEN fato_sk IS NULL THEN 1
                    ELSE 0
                END
             ) = 0
         AND SUM(qtd_beneficios) = COUNT(*)
        THEN 'OK'
        ELSE 'FALHA'
    END AS resultado_validacao_fato,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT fato_sk) AS fato_sk_distintas,
    COUNT(*) - COUNT(DISTINCT fato_sk)
        AS diferenca_chaves,
    SUM(
        CASE
            WHEN fato_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS fato_sk_nulas,
    SUM(qtd_beneficios) AS qtd_beneficios
FROM afastamento_inss.gold.fato_afastamentos;

In [0]:
%sql
-- ============================================================
-- 3. IDENTIFICAR fato_sk DUPLICADAS
-- Resultado esperado: nenhuma linha
-- ============================================================

SELECT
    fato_sk,
    COUNT(*) AS qtd_registros
FROM afastamento_inss.gold.fato_afastamentos
GROUP BY fato_sk
HAVING COUNT(*) > 1
ORDER BY qtd_registros DESC;


In [0]:
%sql
-- ============================================================
-- 4. IDENTIFICAR fato_sk NULAS
-- Resultado esperado: 0
-- ============================================================

SELECT
    COUNT(*) AS qtd_fato_sk_nulas
FROM afastamento_inss.gold.fato_afastamentos
WHERE fato_sk IS NULL;

In [0]:
%sql
-- ============================================================
-- 5. ANALISAR REPETICOES DO HASH-BASE
-- Repeticoes podem existir e nao representam erro automatico
-- ============================================================

SELECT
    fato_hash_base,
    COUNT(*) AS qtd_registros,
    MIN(registro_ocorrencia) AS menor_ocorrencia,
    MAX(registro_ocorrencia) AS maior_ocorrencia
FROM afastamento_inss.gold.fato_afastamentos
GROUP BY fato_hash_base
HAVING COUNT(*) > 1
ORDER BY qtd_registros DESC;

In [0]:
%sql
-- ============================================================
-- 6. RESUMO DOS HASHES-BASE
-- ============================================================

WITH resumo_hash AS (
    SELECT
        fato_hash_base,
        COUNT(*) AS qtd_registros
    FROM afastamento_inss.gold.fato_afastamentos
    GROUP BY fato_hash_base
)
SELECT
    COUNT(*) AS hashes_base_distintos,
    SUM(
        CASE
            WHEN qtd_registros = 1 THEN 1
            ELSE 0
        END
    ) AS hashes_com_um_registro,
    SUM(
        CASE
            WHEN qtd_registros > 1 THEN 1
            ELSE 0
        END
    ) AS hashes_repetidos,
    SUM(
        CASE
            WHEN qtd_registros > 1 THEN qtd_registros
            ELSE 0
        END
    ) AS registros_em_hashes_repetidos,
    MAX(qtd_registros) AS maior_repeticao,
    AVG(qtd_registros) AS media_registros_por_hash
FROM resumo_hash;

In [0]:
%sql
-- ============================================================
-- 7. VALIDAR SEQUENCIA DE registro_ocorrencia
-- Resultado esperado: nenhuma linha
-- ============================================================

WITH validacao_ocorrencia AS (
    SELECT
        fato_hash_base,
        COUNT(*) AS qtd_registros,
        COUNT(DISTINCT registro_ocorrencia) AS ocorrencias_distintas,
        MIN(registro_ocorrencia) AS menor_ocorrencia,
        MAX(registro_ocorrencia) AS maior_ocorrencia
    FROM afastamento_inss.gold.fato_afastamentos
    GROUP BY fato_hash_base
)
SELECT
    fato_hash_base,
    qtd_registros,
    ocorrencias_distintas,
    menor_ocorrencia,
    maior_ocorrencia
FROM validacao_ocorrencia
WHERE ocorrencias_distintas <> qtd_registros
   OR menor_ocorrencia <> 1
   OR maior_ocorrencia <> qtd_registros;

In [0]:
%sql

-- ============================================================
-- VALIDAR CHAVES DIMENSIONAIS NULAS
-- Resultado esperado: zero em todas as colunas
-- ============================================================

SELECT
    SUM(
        CASE
            WHEN tempo_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS tempo_sk_nulas,

    SUM(
        CASE
            WHEN cid_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS cid_sk_nulas,

    SUM(
        CASE
            WHEN especie_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS especie_sk_nulas,

    SUM(
        CASE
            WHEN geografia_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS geografia_sk_nulas,

    SUM(
        CASE
            WHEN atividade_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS atividade_sk_nulas

FROM afastamento_inss.gold.fato_afastamentos;

In [0]:
%sql

-- ============================================================
-- VALIDAR A CHAVE TECNICA DA FATO
-- Resultado esperado:
-- total_registros = fato_sk_distintas
-- fato_sk_nulas = 0
-- ============================================================

SELECT
    COUNT(*) AS total_registros,

    COUNT(DISTINCT fato_sk) AS fato_sk_distintas,

    SUM(
        CASE
            WHEN fato_sk IS NULL THEN 1
            ELSE 0
        END
    ) AS fato_sk_nulas,

    COUNT(*) - COUNT(DISTINCT fato_sk)
        AS diferenca_entre_registros_e_chaves,

    COUNT(DISTINCT fato_hash_base)
        AS hashes_base_distintos,

    MAX(registro_ocorrencia)
        AS maior_registro_ocorrencia,

    SUM(qtd_beneficios)
        AS qtd_beneficios

FROM afastamento_inss.gold.fato_afastamentos;

In [0]:
%sql

-- ============================================================
-- VALIDAR INDICADORES DA FATO
-- ============================================================

SELECT
    COUNT(*) AS total_registros,

    SUM(qtd_beneficios)
        AS qtd_beneficios,

    SUM(ind_saude_mental)
        AS qtd_saude_mental,

    SUM(ind_osteomuscular)
        AS qtd_osteomuscular,

    SUM(ind_acidentario)
        AS qtd_acidentario,

    SUM(ind_afastamento_saude_mental)
        AS qtd_afastamento_saude_mental,

    SUM(ind_afastamento_osteomuscular)
        AS qtd_afastamento_osteomuscular,
    SUM(ind_cardiovascular)
        AS qtd_cardiovascular,
    SUM(ind_respiratorio)
        AS qtd_respiratorio,
    SUM(ind_afastamento_cardiovascular)
        AS qtd_afastamento_cardiovascular,
    SUM(ind_afastamento_respiratorio)
        AS qtd_afastamento_respiratorio

FROM afastamento_inss.gold.fato_afastamentos;

In [0]:
%sql

-- ============================================================
-- VALIDAR A SEQUENCIA DE OCORRENCIAS POR HASH
-- Resultado esperado: nenhuma linha
-- ============================================================

WITH validacao_ocorrencia AS (
    SELECT
        fato_hash_base,

        COUNT(*) AS qtd_registros,

        COUNT(DISTINCT registro_ocorrencia)
            AS qtd_ocorrencias_distintas,

        MIN(registro_ocorrencia)
            AS menor_ocorrencia,

        MAX(registro_ocorrencia)
            AS maior_ocorrencia

    FROM afastamento_inss.gold.fato_afastamentos

    GROUP BY fato_hash_base
)

SELECT
    fato_hash_base,
    qtd_registros,
    qtd_ocorrencias_distintas,
    menor_ocorrencia,
    maior_ocorrencia

FROM validacao_ocorrencia

WHERE qtd_ocorrencias_distintas <> qtd_registros
   OR menor_ocorrencia <> 1
   OR maior_ocorrencia <> qtd_registros;


In [0]:
%sql

-- ============================================================
-- IDENTIFICAR CHAVES fato_sk DUPLICADAS
-- Resultado esperado: nenhuma linha
-- ============================================================

SELECT
    fato_sk,
    COUNT(*) AS qtd_registros

FROM afastamento_inss.gold.fato_afastamentos

GROUP BY fato_sk

HAVING COUNT(*) > 1

ORDER BY qtd_registros DESC;